In [ ]:

%pip -q install -U --no-deps \
    "huggingface-hub==0.35.3" \
    "tokenizers==0.21.1" \
    "transformers==4.51.3" \
    "datasets==3.6.0" \
    "accelerate==1.1.1" \
    "safetensors>=0.4.3"

print("Готово")
raise SystemExit("Restart runtime required")

Готово. Теперь: Runtime -> Restart session, затем запуск с ячейки 2.


SystemExit: Restart runtime required

In [14]:
import os
import re
import random
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
import transformers, tokenizers, huggingface_hub

from pathlib import Path
from datasets import Dataset
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score, precision_recall_fscore_support,
    roc_auc_score, average_precision_score,
    confusion_matrix, classification_report,
    matthews_corrcoef
)
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    DataCollatorWithPadding, TrainingArguments,
    Trainer, EarlyStoppingCallback
)

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

MODEL_NAME = "microsoft/deberta-v3-large"
MAX_LENGTH = 128

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Tokenizers:", tokenizers.__version__)
print("HF Hub:", huggingface_hub.__version__)
print("CUDA available:", torch.cuda.is_available())

Torch: 2.10.0+cu128
Transformers: 4.51.3
Tokenizers: 0.21.1
HF Hub: 0.35.3
CUDA available: True


In [ ]:
# Пути к файлам. При необходимости поменяйте под ваш диск в Colab.
CRYPTO_PATH = "crypto_twitter_dataset2.csv"
NON_CRYPTO_PATH = "non_crypto_tweets.csv"

def safe_read_csv(path):
    p = Path(path)
    if not p.exists():
        candidates = list(Path('.').glob(f"**/{p.name}"))
        if not candidates:
            raise FileNotFoundError(f"Не найден файл: {path}")
        p = candidates[0]

    try:
        df = pd.read_csv(
            p,
            sep=';',
            engine="python",
            on_bad_lines="skip",
            quotechar='"',
            encoding="utf-8"
        )
    except Exception:
        df = pd.read_csv(
            p,
            sep=None,
            engine="python",
            on_bad_lines="skip",
            quotechar='"',
            encoding="utf-8"
        )

    if df.empty:
        raise ValueError(f"Файл {p} прочитан, но данных нет.")

    return df

crypto_df = safe_read_csv(CRYPTO_PATH)
non_crypto_df = safe_read_csv(NON_CRYPTO_PATH)

print("Crypto shape:", crypto_df.shape)
print("Non-crypto shape:", non_crypto_df.shape)
print("Crypto columns:", list(crypto_df.columns)[:10], "...")
print("Non-crypto columns:", list(non_crypto_df.columns))

Crypto shape: (24038, 41)
Non-crypto shape: (74915, 7)
Crypto columns: ['followers_count', 'friends_count', 'followers_friends_ratio', 'statuses_count', 'favourites_count', 'listed_count', 'media_count', 'account_age_days', 'tweets_per_day', 'has_custom_timelines'] ...
Non-crypto columns: ['tweet_id', 'created_at', 'full_text', 'favorite_count', 'retweet_count', 'reply_count', 'lang']


In [16]:
# Нормализация схемы данных
TEXT_CANDIDATES = ["tweet_text", "full_text", "text", "tweet"]
ID_CANDIDATES = ["tweet_id", "id"]

def first_existing_column(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

def normalize_df(df, label):
    text_col = first_existing_column(df, TEXT_CANDIDATES)
    if text_col is None:
        raise ValueError(f"Не найден текстовый столбец. Ожидались: {TEXT_CANDIDATES}")

    id_col = first_existing_column(df, ID_CANDIDATES)

    out = pd.DataFrame({
        "text": df[text_col].astype(str),
        "label": int(label)
    })
    if id_col is not None:
        out["tweet_id"] = df[id_col].astype(str)
    else:
        out["tweet_id"] = np.arange(len(out)).astype(str)
    return out

crypto_norm = normalize_df(crypto_df, label=1)
non_crypto_norm = normalize_df(non_crypto_df, label=0)
data = pd.concat([crypto_norm, non_crypto_norm], ignore_index=True)

def clean_text(text):
    text = text.lower()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"@\w+", " ", text)
    text = re.sub(r"#(\w+)", r"\1", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

data["text"] = data["text"].fillna("").map(clean_text)
data = data[data["text"].str.len() >= 3].drop_duplicates(subset=["text"]).reset_index(drop=True)

print(data.head(3))
print("Total samples:", len(data))
print("Class distribution:", data["label"].value_counts(normalize=True).sort_index())

                                                text  label  \
0  bitcoin semakin naik mendekati resistance gari...      1   
1  📊 xrp weekly fib update! shoutout to our stude...      1   
2                       bitcoin just broke $93,000 🚀      1   

              tweet_id  
0  1994414150211326224  
1  1994405935842840576  
2  1994413580071174424  
Total samples: 88532
Class distribution: label
0    0.742184
1    0.257816
Name: proportion, dtype: float64


In [17]:
# Train / validation / test split (stratified)
train_df, temp_df = train_test_split(
    data, test_size=0.2, random_state=SEED, stratify=data["label"]
)
val_df, test_df = train_test_split(
    temp_df, test_size=0.5, random_state=SEED, stratify=temp_df["label"]
)

print("Train:", train_df.shape, "Val:", val_df.shape, "Test:", test_df.shape)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_batch(batch):
    return tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH)

train_ds = Dataset.from_pandas(train_df[["text", "label"]], preserve_index=False)
val_ds = Dataset.from_pandas(val_df[["text", "label"]], preserve_index=False)
test_ds = Dataset.from_pandas(test_df[["text", "label"]], preserve_index=False)

train_ds = train_ds.map(tokenize_batch, batched=True)
val_ds = val_ds.map(tokenize_batch, batched=True)
test_ds = test_ds.map(tokenize_batch, batched=True)

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

Train: (70825, 3) Val: (8853, 3) Test: (8854, 3)


/usr/local/lib/python3.12/dist-packages/transformers/convert_slow_tokenizer.py:559: UserWarning: The sentencepiece tokenizer that you are converting to a fast tokenizer uses the byte fallback option which is not implemented in the fast tokenizers. In practice this means that the fast version of the tokenizer can produce unknown tokens whereas the sentencepiece version would have converted these unknown tokens into a sequence of byte tokens matching the original piece of text.
  warnings.warn(


Map:   0%|          | 0/70825 [00:00<?, ? examples/s]

Map:   0%|          | 0/8853 [00:00<?, ? examples/s]

Map:   0%|          | 0/8854 [00:00<?, ? examples/s]

In [18]:
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    probs = torch.softmax(torch.tensor(logits), dim=-1).numpy()
    preds = np.argmax(logits, axis=-1)

    precision, recall, f1, _ = precision_recall_fscore_support(
        labels, preds, average="binary", zero_division=0
    )
    acc = accuracy_score(labels, preds)
    roc_auc = roc_auc_score(labels, probs[:, 1])
    pr_auc = average_precision_score(labels, probs[:, 1])
    mcc = matthews_corrcoef(labels, preds)

    return {
        "accuracy": float(acc),
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "roc_auc": float(roc_auc),
        "pr_auc": float(pr_auc),
        "mcc": float(mcc)
    }

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError(
        "GPU не активен в PyTorch."
    )

try:
    import httpx
    _orig_httpx_head = httpx.Client.head

    def _patched_httpx_head(self, url, *args, **kwargs):
        if "allow_redirects" in kwargs and "follow_redirects" not in kwargs:
            kwargs["follow_redirects"] = kwargs.pop("allow_redirects")
        return _orig_httpx_head(self, url, *args, **kwargs)

    httpx.Client.head = _patched_httpx_head
except Exception:
    pass

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2,
    use_safetensors=False
)

if hasattr(model.config, "use_cache"):
    model.config.use_cache = False

eval_strategy = "epoch"

training_args = TrainingArguments(
    output_dir="./deberta_crypto_detector",
    learning_rate=1.2e-5,
    num_train_epochs=3,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=8,
    gradient_accumulation_steps=4,
    warmup_ratio=0.1,
    weight_decay=0.01,
    eval_strategy=eval_strategy,
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    metric_for_best_model="f1",
    greater_is_better=True,
    logging_steps=50,
    report_to="none",
    fp16=True,
    bf16=False,
    gradient_checkpointing=False,
    dataloader_pin_memory=True,
    seed=SEED
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)]
)

trainer.train()

Some weights of DebertaV2ForSequenceClassification were not initialized from the model checkpoint at microsoft/deberta-v3-large and are newly initialized: ['classifier.bias', 'classifier.weight', 'pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
/tmp/ipykernel_7135/4149628257.py:57: FutureWarning: `tokenizer` is deprecated and will be removed in version 5.0.0 for `Trainer.__init__`. Use `processing_class` instead.
  trainer = Trainer(


Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Accuracy,Precision,Recall,F1,Roc Auc,Pr Auc,Mcc
0,0.192500,0.202381,0.938439,0.941985,0.811131,0.871674,0.953614,0.930181,0.835501
1,0.138100,0.201069,0.942844,0.957261,0.814636,0.880208,0.961082,0.938340,0.847726


TrainOutput(global_step=8852, training_loss=0.1960408566490159, metrics={'train_runtime': 7207.4865, 'train_samples_per_second': 19.653, 'train_steps_per_second': 1.228, 'total_flos': 1.5024407639910528e+16, 'train_loss': 0.1960408566490159, 'epoch': 1.9998305754786243})

In [20]:
# Итоговая оценка на test
test_metrics = trainer.evaluate(test_ds)
print("Test metrics:")
for k, v in test_metrics.items():
    if isinstance(v, float):
        print(f"{k}: {v:.4f}")
    else:
        print(f"{k}: {v}")

pred_out = trainer.predict(test_ds)
logits = pred_out.predictions
y_true = pred_out.label_ids
y_prob = torch.softmax(torch.tensor(logits), dim=-1).numpy()[:, 1]
y_pred = np.argmax(logits, axis=-1)

print("\nClassification report:")
print(classification_report(y_true, y_pred, digits=4, target_names=["non_crypto", "crypto"]))

Test metrics:
eval_loss: 0.1857
eval_accuracy: 0.9447
eval_precision: 0.9498
eval_recall: 0.8292
eval_f1: 0.8854
eval_roc_auc: 0.9667
eval_pr_auc: 0.9461
eval_mcc: 0.8526
eval_runtime: 65.0329
eval_samples_per_second: 136.1460
eval_steps_per_second: 17.0220
epoch: 1.9998

Classification report:
              precision    recall  f1-score   support

  non_crypto     0.9432    0.9848    0.9635      6571
      crypto     0.9498    0.8292    0.8854      2283

    accuracy                         0.9447      8854
   macro avg     0.9465    0.9070    0.9245      8854
weighted avg     0.9449    0.9447    0.9434      8854



In [ ]:
# Визуализация Confusion Matrix (без seaborn)
cm = confusion_matrix(y_true, y_pred)
labels = ["non_crypto", "crypto"]

plt.figure(figsize=(6, 5))
plt.imshow(cm, cmap="Blues")
plt.title("Confusion Matrix")
plt.colorbar()

tick_marks = np.arange(len(labels))
plt.xticks(tick_marks, labels)
plt.yticks(tick_marks, labels)
plt.xlabel("Predicted")
plt.ylabel("True")

for i in range(cm.shape[0]):
    for j in range(cm.shape[1]):
        plt.text(j, i, str(cm[i, j]), ha="center", va="center", color="black")

plt.tight_layout()
plt.show()

In [ ]:
# Сохранение модели и токенайзера + архив для скачивания на компьютер + копия на Google Drive
import os
import shutil

SAVE_DIR = "./deberta_crypto_detector_best"
trainer.save_model(SAVE_DIR)
tokenizer.save_pretrained(SAVE_DIR)
print("Saved to:", SAVE_DIR)

ARCHIVE_BASE = "./deberta_crypto_detector_best"
ARCHIVE_PATH = shutil.make_archive(ARCHIVE_BASE, "zip", SAVE_DIR)
print("Archive created:", ARCHIVE_PATH)

try:
    from google.colab import files
    files.download(ARCHIVE_PATH)
    print("Download started in browser.")
except Exception:
    print("Not running in Colab. Archive is ready at:", os.path.abspath(ARCHIVE_PATH))

try:
    from google.colab import drive
    drive.mount("/content/drive", force_remount=False)

    gdrive_root = "/content/drive/MyDrive/deberta_crypto_exports"
    os.makedirs(gdrive_root, exist_ok=True)

    # Копируем папку модели
    gdrive_model_dir = os.path.join(gdrive_root, "deberta_crypto_detector_best")
    if os.path.exists(gdrive_model_dir):
        shutil.rmtree(gdrive_model_dir)
    shutil.copytree(SAVE_DIR, gdrive_model_dir)

    # Копируем zip-архив
    gdrive_archive_path = os.path.join(gdrive_root, os.path.basename(ARCHIVE_PATH))
    shutil.copy2(ARCHIVE_PATH, gdrive_archive_path)

    print("Saved to Google Drive:")
    print("Model folder:", gdrive_model_dir)
    print("Archive:", gdrive_archive_path)
except Exception as e:
    print("Google Drive save skipped:", repr(e))

NameError: name 'trainer' is not defined

In [23]:
# Пример инференса
id2label = {0: "non_crypto", 1: "crypto"}

def predict_tweet(text, model, tokenizer, max_length=128):
    model.eval()
    enc = tokenizer(text, return_tensors="pt", truncation=True, max_length=max_length)
    if torch.cuda.is_available():
        enc = {k: v.cuda() for k, v in enc.items()}
        model.cuda()

    with torch.no_grad():
        outputs = model(**enc)
        probs = torch.softmax(outputs.logits, dim=-1).cpu().numpy()[0]

    pred = int(np.argmax(probs))
    return {
        "label": id2label[pred],
        "crypto_probability": float(probs[1]),
        "non_crypto_probability": float(probs[0])
    }

sample = "Bitcoin ETF inflows hit a new high this week, market looks bullish."
print(predict_tweet(sample, model, tokenizer))

{'label': 'crypto', 'crypto_probability': 0.9994207620620728, 'non_crypto_probability': 0.0005792917218059301}
